In [1]:
import sys
import os

# Adds 'src/capstone' to the path so 'pipeline' is discoverable
# assuming the notebook is in ROOT/notebooks/ and code is in ROOT/src/capstone/
sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

`use_synthetic=True, target_real_pct=5/6` → SyntheticAugmenter appends ~20% synthetic
rows to `X_train` (formula: `floor(real / target_real_pct) - real`).
No snapshot flags set → `build()` leaves all version counters unchanged.
This run will read from GCS but write nothing back.

In [2]:
config = VersionConfig.load(use_synthetic=True, target_real_pct=5/6).build()
run    = PipelineRun(config)
stages = PipelineFactory.retrain_existing_data(config)

print('\nScenario:', stages.scenario)
print('raw_version   :', config.raw_version)
print('model_version :', config.model_version)

VersionConfig loaded:
  data:          v3.2 (raw_suffix='real')
  model:         v3.1
  hyperparams:   v1.0
  use_synthetic: True (target_real_pct=83%)

VersionConfig ready:
  Active flags      : ['none (dry run)']
  data              : v3.2 (unchanged)
  model             : v3.1 (unchanged)
  hyperparams       : v1.0 (unchanged)
  raw_version       : v3.2_real
  final_version     : v3.2_mixed_83real
  model_version     : v3.1
  hyperparam_version: v1.0
  use_synthetic     : True

Scenario: retrain_existing_data
raw_version   : v3.2_real
model_version : v3.1


## First-time Setup: Create the Locked Validation Holdout

**Run once**, before any pipeline scenario. Skip if `splits/validation_ids.json` already
exists in GCS — `DataSplitter` detects the file and errors helpfully if it's missing.

```bash
python scripts/create_validation_set.py        # dry-run (default)
python scripts/create_validation_set.py --yes  # write to GCS
```

The script runs Load → DataPreprocessor → FeatureEngineer, then creates a stratified
holdout using the 18-cell key (vertical × tier × above_baseline).

---
## Main Flow

Runs the full `retrain_existing_data` scenario with synthetic augmentation.

## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. No BigQuery call.

In [3]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.2_real': 58037 rows from 2026-04-27
  Polls: {'upload': 20468, '24h': 20048, '7d': 17521}
Loaded baselines 'v3.2_real': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  y_train        None
  y_test         None
  y_val          None
  models         empty dict
  results        empty dict


## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video (via `pivot_snapshots`),
joins channel baselines, and runs structural cleanup via `build_clean_dataset`.
Writes `run.df_clean`.

In [4]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 17463 (dropped 3052 incomplete)
  Pivoted shape: (17532, 34)

[2/3] Joining baseline medians...
  Baseline join: 17532/17532 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 17532 rows × 40 columns

Clean dataset: 17532 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  y_train        None
  y_test         None
  y_val          None
  models         empty dict
  results        empty dict


## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean` (one row per video).
Computes `above_baseline` target, all ratio/growth features, and categorical encodings.
Writes `run.df_engineered`.

In [5]:
stages.engineer.run(run)
run.summary()

  all: dropped 396 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/9] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/9] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/9] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/9] Computing categorical features...
  Title categories:
title_category
neutral        10835
all_caps        1931
exclamation     1861
question        1498
listicle         527
how_to           219
clickbait        193
emoji_heavy       72
  Description categories:
desc_category
link_heavy        4519
minimal           4297
has_links         3656
has_hashtags      2559
neutral           1323
has_timestamps     736
long_form           4

## Stage 4 — DataSplitter

Loads the locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows into `df_train` / `df_test` (80/20). Derives unscaled `X_train`, `X_test`,
`X_val` and corresponding `y_` series.

In [6]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (30.3%)
  df_train:  9,554 rows (55.8%)
  df_test:   2,389 rows (13.9%)
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  None
  y_train        populated  Series length=9554
  y_test         populated  Series length=2389
  y_val          pop

In [7]:
# Debugging
#print(run.df_train.columns)


## Stage 5 — Scaler

Fits `StandardScaler` on `X_train`, transforms `X_train`, `X_test`, and `X_val`.
Captures `run.X_val_unscaled` so Validator can apply each loaded model's own
historical scaler for a fair apples-to-apples comparison.

In [8]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=9554
  y_test         populated  Series length=2389
  y_val          populated  Series length=5193
  models         empty dict
  results        empty dict


## Stage 6 — SyntheticAugmenter

Generates synthetic rows from `run.df_train` via `generate_synthetic_data`,
engineers them through the same `FeatureEngineerLogic` chain, aligns with
`X_train`'s column space, scales with the fitted `StandardScaler`, then
concatenates onto `run.X_train` / `run.y_train`. `X_test` and `X_val` are
never touched. Target: ~20% of real train rows added as synthetic
(`target_real_pct=5/6` → `floor(real/0.8333) - real ≈ 0.2 * real`).

In [9]:
stages.augmenter.run(run)
run.summary()

Generating 1910 synthetic rows

[1/5] Preparing data for synthesis...
  Input shape: (9554, 22)

[2/5] Fitting GaussianCopulaSynthesizer...


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


  Fit complete

[3/5] Sampling synthetic rows...
  Generated 1910 rows

[4/5] Assigning real channels and baselines...
  Assigned 1910 synthetic rows to 542 real channels
  Baseline match: 1910/1910

[5/5] Postprocessing and recomputing features...

Synthetic data: 1910 rows
Assigned to 542 real channels
Target balance: {1: 1060, 0: 850}
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.5% above baseline, 44.5% below

[2/9] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/9] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/9] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/9] Computing categorical features...
  Title categories:
title_category
neutral    1910
  Description categories:
desc_category
minimal    1910

[6/9] Computing text structural features...
  Computed t

/Users/jelanigould-bailey/Desktop/uc_berkeley_capstone_repo/repo/src/capstone/pipeline/stages/feature_engineer.py:155: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols] = df[cols].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
/Users/jelanigould-bailey/Desktop/uc_berkeley_capstone_repo/repo/src/capstone/pipeline/stages/feature_engineer.py:155: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols] = df[cols].repl

## Stage 7 — Scaler
Scale the data after splitting, for validation.

In [10]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(11464, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=11464
  y_test         populated  Series length=2389
  y_val          populated  Series length=5193
  models         empty dict
  results        empty dict


## Stage 8 — ModelLoader

Auto-discovers all `models/<model_version>_*/` directories in GCS and loads each
saved model, scaler, and feature column list into `run.models`.

In [11]:
stages.model_loader.run(run)
print('\nLoaded models:', list(run.models.keys()))

Loading 3 models for base version 'v3.1': ['v3.1_lr_l1', 'v3.1_rf', 'v3.1_xgb']
Loaded model 'v3.1_lr_l1' (LogisticRegression)
  ROC-AUC: 0.7715
Loaded model 'v3.1_rf' (RandomForest)
  ROC-AUC: 0.8517
Loaded model 'v3.1_xgb' (XGBoost)
  ROC-AUC: 0.884

Loaded models: ['lr_l1', 'rf', 'xgb']


## Stage 8 — Validator

Evaluates each loaded model on the locked validation set (`X_val_unscaled` +
each model's own historical scaler). Populates `run.results` with AUC, accuracy,
F1, precision, recall, and top feature importances per model.

In [12]:
stages.validator.run(run)
run.summary()

[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']
[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']
[Validator] 1 feature(s) absent from X_val — mean-imputed: ['hours_since_publish_24h']

=== Validator — validation-set results ===
  lr_l1           AUC=0.7003  acc=0.6332  F1↑=0.6300
  rf              AUC=0.5980  acc=0.5738  F1↑=0.7184
  xgb             AUC=0.6587  acc=0.5837  F1↑=0.7115
PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame

/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


## Stage 9 — Persist Validation Results to GCS

In [13]:
stages.validation_results_snapshotter.run(run)
# Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl

Validation results appended → gs://maduros-dolce-capstone-data/models/v3.1/validation_results.jsonl
  lr_l1           AUC=0.7003  acc=0.6332  F1↑=0.6300
  rf              AUC=0.5980  acc=0.5738  F1↑=0.7184
  xgb             AUC=0.6587  acc=0.5837  F1↑=0.7115


PipelineRun(config=VersionConfig(data=(3, 2), model=(3, 1), hyperparams=(1, 0), use_synthetic=True, active=[]), df_videos=          video_id                   poll_timestamp                channel_id  \
0      R-aVXzh_Szs 2026-04-25 19:05:06.990886+00:00  UCV-yCAXMS-nWrHtORuDfSuQ   
1      SxZFQn8s0us 2026-04-25 19:05:06.990886+00:00  UCzkZKIaXd9CP2_l4u3Tmmmg   
2      M62BbuNvriI 2026-04-25 19:05:06.990886+00:00  UCFjc9H89-RpWuIStDqhO7AQ   
3      dBbhvhjtAWE 2026-04-25 13:05:09.820304+00:00  UCsLF0qPTpkYKq81HsjgzhwQ   
4      eSX2C1LJLGs 2026-04-25 13:05:09.820304+00:00  UCZtR1d3RYi29s_rlRNJhKrg   
...            ...                              ...                       ...   
58032  CjoSc7yZVuQ 2026-04-05 22:05:05.884862+00:00  UCPKEknmUB78oPHG_07VZVXQ   
58033  Cq72IsHBolY 2026-04-06 16:05:26.236288+00:00  UC4Bj2n_0vzs4tPrggk_yhgw   
58034  vIXXq26iPPg 2026-04-05 07:05:16.597323+00:00  UC4Bj2n_0vzs4tPrggk_yhgw   
58035  Cq72IsHBolY 2026-04-05 19:05:05.109837+00:00  UC4Bj2n_0vzs4tP

## Results

This is the **first validation-set evaluation** for this project — there is no prior
validation baseline in `data_analysis.ipynb` (only test-set scores were recorded there).

Cross-check model rank order and approximate AUC range against the test scores
from `data_analysis.ipynb`. Validation AUC will typically be slightly lower than
test AUC since the holdout was never touched during training.

In [14]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

,roc_auc,accuracy,f1_above,precision_above,recall_above,f1_below,precision_below,recall_below
model,,,,,,,,
lr_l1,0.7003,0.6332,0.6300,0.7161,0.5624,0.6362,0.5690,0.7215
rf,0.5980,0.5738,0.7184,0.5674,0.9788,0.1243,0.7202,0.0680
xgb,0.6587,0.5837,0.7115,0.5783,0.9244,0.2524,0.6261,0.1581


In [15]:
# Top features for each model — sanity check against data_analysis.ipynb
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')


lr_l1:
  like_rate_24h                        +3.8403
  view_count_upload_vs_baseline        -2.1137
  view_count_24h                       -1.7407
  like_count_24h                       +1.1326
  view_velocity_upload                 +0.8387

rf:
  like_rate_24h                        +0.0785
  view_count_24h                       +0.0494
  view_count_velocity_24h              +0.0438
  views_per_sub_24h                    +0.0416
  likes_per_sub_24h                    +0.0402

xgb:
  baseline_baseline_video_count        +0.0694
  like_rate_24h                        +0.0462
  like_count_24h                       +0.0353
  view_count_24h                       +0.0329
  is_short                             +0.0321


# Final: GCS Write 

Commit config changes to GCS.

In [16]:
config.commit()

No snapshots were active — nothing to commit.
